# **CABAgent: <ins>C</ins>omprehensive Layout-Aware <ins>A</ins>nalog <ins>B</ins>enchmark Generation via Self-Improving LLM <ins>Agent</ins>s for Analog Circuit Design Automation**

| Name                    | Affiliation                                                                                          | IEEE Member | SSCS Member |
|:-----------------------:|:----------------------------------------------------------------------------------------------------:|:-----------:|:-----------:|
| Jinhai Hu*              | Institute of Microelectronics, A*STAR, Singapore                                                     | Yes         | Yes         |
| Jiageng Wang*           | Nanyang Technological University, Singapore                                                          | No          | No          |
| Xinzhe Xie              | Nanyang Technological University, Singapore; <br /> Institute of Microelectronics, A*STAR, Singapore | Yes         | Yes         |
| Wang Ling Goh           | Nanyang Technological University, Singapore                                                          | Yes         | Yes         |
| Zhuoyi Lin              | Institute for Infocomm Research, A*STAR, Singapore                                                   | Yes         | No          |
| Yuan Gao                | Institute of Microelectronics, A*STAR, Singapore                                                     | Yes         | Yes         |

## Abstract

This project presents 

<hr style="border:2px solid grey">

## Introduction

This repository  

<hr style="border:2px solid grey">

## Architecture

<hr style="border:2px solid grey">

## Step 1: LLM-Driven Netlist Generation (AnalogAgent)

AnalogAgent takes a **natural-language circuit description** and generates a **SKY130 PDK-compatible SPICE subcircuit netlist** through an agentic loop:

1. **Code Generator** — LLM generates a `.subckt` netlist from prompt + SEM guidance
2. **Design Optimizer** — Runs static checks and ngspice DC validation; reflects on failures
3. **Knowledge Curator** — Distills failure/success into reusable rules (Self-Evolving Memory)

The generated netlist is then post-processed (device renaming, param splitting) and fed into the CABGen pipeline below.

<hr style="border:2px solid grey">

In [ ]:
from src.analogagent import generate_netlist

In [ ]:
# --- Generate SKY130 netlist for Five-Transistor OTA ---
result_ota5t = generate_netlist(
    task="a five-transistor OTA, with differential input and single output, "
         "using 1:1 current mirror to provide tail current",
    pins="VDD, VSS, VIN, VIP, VOUT, IB",
    output_node="VOUT",
    output_dir="designs/OTA_5T/SKY130/inputs",
    max_retries=5,
)

print(f"\nSuccess: {result_ota5t['success']}")
print(f"Iterations used: {result_ota5t['iterations']}")
if result_ota5t['raw_code']:
    print(f"\n--- Generated Netlist ---")
    print(result_ota5t['raw_code'])

## Step 2: Layout-Aware Benchmark Generation (CABGen)

With the generated netlist in place, the CABGen pipeline takes over:

1. **Pre-simulation** — Embed netlist into testbench, run Ngspice AC/noise/transient analysis
2. **Layout generation** — Convert netlist to ALIGN format, generate GDS layout
3. **Verification** — DRC (KLayout), parasitic extraction (Magic), LVS (Netgen)
4. **Post-simulation** — Re-run simulations with extracted parasitics

This is repeated across multiple parameter combinations to build a comprehensive benchmark.

<hr style="border:2px solid grey">

In [3]:
from src.design_pipeline import EDAPipeline

In [4]:
DEFAULT_PDK = "SKY130"

DEFAULT_STEPS = {
    "pre-sim"   : "Ngspice",
    "layout"    : "ALIGN",
    "drc"       : "KLayout",
    "extract"   : "Magic",
    "lvs"       : "Netgen",
    "post-sim"  : "Ngspice",
}

DEFAULT_RESET = {
    "keep_top"  : {"align", "klayout", "magic", "netgen", "ngspice"},
    "keep_path" : {"align": {"0_netlist"}, "ngspice": {".spiceinit", "param.spice"}},
    "verbose"   : False,
}

In [5]:
pipeline = EDAPipeline("OTA_5T", pdk=DEFAULT_PDK, steps=DEFAULT_STEPS, reset_kwargs=DEFAULT_RESET)

Design configuration loaded:
 design  :
   name       : OTA_5T
   circuit    : five_transistor_ota
   pin_order  : VDD VSS VIN VIP VOUT IB
   top_module : FIVE_TRANSISTOR_OTA_0
 paths   :
   root : designs/OTA_5T/SKY130
 inputs  :
   netlist_file : /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/inputs/ckt_netlist.spice
   param_file   : /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/inputs/ckt_param.spice
   const_file   : /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/inputs/constraints.json
   tb_file      : /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/inputs/testbench.spice
 align   :
   input_dir  : /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/runs/align/0_netlist
   sky130_dir : /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks

In [6]:
pipeline.run_cabgen(num_trials=10)

INFO: PIPELINE: Starting Benchmark Generation for OTA_5T


Preserved: /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/runs/ngspice/param.spice
Preserved: /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/runs/ngspice/.spiceinit
Preserved: /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/runs/align/0_netlist
Replaced: align input netlist with CAB-generated param.spice
INFO: RESET: Workspace reset for benchmark generation trail 0
[PRE-SIM] → Ngspice
INFO: NGSPICE: ngspice -b /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/runs/ngspice/five_transistor_ota_presim.spice
INFO: NGSPICE: Simulation Finished! Please check logs/ngspice.log for details.
[LAYOUT] → ALIGN
INFO: ALIGN: schematic2layout.py /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_notebooks/CABAgent/designs/OTA_5T/SKY130/runs/align/0_netlist -p /home/sscs-ose-code-a-chip.github.io/VLSI26/submitted_

## Results and Discussion

<hr style="border:2px solid grey">

## Conclusion



<hr style="border:2px solid grey">

## References